# Training Log Analysis - Console Output Parser

This notebook analyzes **console output logs** from training runs to extract:
- Game progression (steps, rounds, turns per game)
- Curriculum phase transitions and their effects
- Warning and error patterns
- Model update tracking
- Training speed and completion statistics

**Complementary to**: `analyze_curriculum_training.ipynb` (analyzes JSON checkpoints)

---

## Quick Start

1. Run training with logging:
   ```bash
   python train.py 2>&1 | Tee-Object -FilePath logs/training_$(Get-Date -Format 'yyyyMMdd_HHmmss').log
   ```

2. Update the `LOG_FILE` path in the cell below

3. Run all cells (Cell → Run All)

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Tuple
from collections import defaultdict
from datetime import datetime

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (16, 8)
plt.rcParams['font.size'] = 10

print("✓ Imports loaded")

## Configuration

In [ ]:
# ============================================================
# CONFIGURATION - Update this path to your log file
# ============================================================

# Option 1: Specific log file
LOG_FILE = Path('logs/training_20250102_143022.log')

# Option 2: Latest log file (uncomment to use)
# log_dir = Path('logs')
# LOG_FILE = sorted(log_dir.glob('training_*.log'))[-1]

# Verify file exists
if LOG_FILE.exists():
    file_size_mb = LOG_FILE.stat().st_size / (1024 * 1024)
    print(f"✓ Log file found: {LOG_FILE.name}")
    print(f"  Size: {file_size_mb:.2f} MB")
else:
    print(f"❌ Log file not found: {LOG_FILE}")
    print(f"   Please update LOG_FILE path or run training with logging first.")

## Log Parser Implementation

In [ ]:
@dataclass
class GameEvent:
    """Single game from training logs."""
    game_num: int
    participants: List[str]
    num_players: int
    winner: Optional[str] = None
    rounds: Optional[int] = None
    turns: Optional[int] = None
    steps: Optional[int] = None
    phase: Optional[int] = None
    max_rounds: Optional[int] = None
    max_actions_multiplier: Optional[int] = None
    warnings: List[str] = field(default_factory=list)
    errors: List[str] = field(default_factory=list)
    step_samples: List[Tuple[int, int, int, int]] = field(default_factory=list)

@dataclass
class PhaseTransition:
    """Phase transition event."""
    game_num: int
    from_phase: int
    to_phase: int
    old_rounds: int
    new_rounds: int
    old_actions: str
    new_actions: str

@dataclass
class ModelUpdate:
    """Model update event."""
    game_num: int
    agents_updated: List[str]
    buffer_sizes: Dict[str, int]
    losses: Dict[str, Dict[str, float]]

print("✓ Data structures defined")

In [ ]:
class LogParser:
    """Parse training log files."""
    
    # Regex patterns
    GAME_START = re.compile(r'\[Game (\d+)\] Starting with (\d+) players?: (.+)')
    GAME_END = re.compile(r'\[Game (\d+)\] Finished! Winner: (\w+), Rounds: (\d+), Turns: (\d+), Steps: (\d+)')
    STEP_PROGRESS = re.compile(r'Step (\d+), Round (\d+), Turn (\d+), (\d+)/(\d+) players active')
    PHASE_TRANSITION = re.compile(
        r'CURRICULUM PHASE TRANSITION: Phase (\d+) → Phase (\d+)\s*\n'
        r'.*Increasing max_rounds: (\d+) → (\d+)\s*\n'
        r'.*Increasing max_actions_per_turn: (.+) → (.+)',
        re.MULTILINE
    )
    MODEL_UPDATE_START = re.compile(r'\[Game (\d+)\] Updating models...')
    MODEL_UPDATE_AGENT = re.compile(r'Updating (\w+)...')
    MODEL_BUFFER_SIZE = re.compile(r'Buffer size: (\d+)')
    MODEL_LOSSES = re.compile(r'(\w+): policy_loss=([\d.]+), value_loss=([\d.]+), entropy=([\d.]+)')
    WARNING = re.compile(r'WARNING: (.+)')
    ERROR = re.compile(r'ERROR: (.+)')
    GAME_ENDING = re.compile(r'Game ending: (.+)')
    
    def __init__(self, log_file: Path):
        self.log_file = log_file
        self.games = []
        self.phase_transitions = []
        self.model_updates = []
        self.current_game = None
        self.current_update = None
        self.current_update_agent = None
    
    def _get_phase(self, game_num: int) -> int:
        """Determine curriculum phase."""
        if game_num < 1000:
            return 1
        elif game_num < 3000:
            return 2
        else:
            return 3
    
    def parse(self):
        """Parse the log file."""
        print(f"Parsing {self.log_file.name}...")
        
        with open(self.log_file, 'r', encoding='utf-8', errors='ignore') as f:
            content = f.read()
        
        lines = content.split('\n')
        
        for i, line in enumerate(lines):
            self._parse_line(line, lines, i)
        
        # Finalize current game
        if self.current_game:
            self.games.append(self.current_game)
        
        print(f"✓ Parsed {len(self.games)} games")
        print(f"  Phase transitions: {len(self.phase_transitions)}")
        print(f"  Model updates: {len(self.model_updates)}")
        
        return self
    
    def _parse_line(self, line: str, all_lines: List[str], line_idx: int):
        """Parse a single line."""
        # Game start
        match = self.GAME_START.search(line)
        if match:
            if self.current_game:
                self.games.append(self.current_game)
            
            game_num = int(match.group(1))
            num_players = int(match.group(2))
            participants = [p.strip() for p in match.group(3).split(',')]
            phase = self._get_phase(game_num)
            
            self.current_game = GameEvent(
                game_num=game_num,
                participants=participants,
                num_players=num_players,
                phase=phase,
                max_rounds={1: 15, 2: 25, 3: 50}[phase],
                max_actions_multiplier={1: 10, 2: 20, 3: 50}[phase]
            )
            return
        
        # Game end
        match = self.GAME_END.search(line)
        if match and self.current_game:
            self.current_game.winner = match.group(2)
            self.current_game.rounds = int(match.group(3))
            self.current_game.turns = int(match.group(4))
            self.current_game.steps = int(match.group(5))
            return
        
        # Step progress
        match = self.STEP_PROGRESS.search(line)
        if match and self.current_game:
            step = int(match.group(1))
            round_num = int(match.group(2))
            turn = int(match.group(3))
            active = int(match.group(4))
            self.current_game.step_samples.append((step, round_num, turn, active))
            return
        
        # Warnings
        match = self.WARNING.search(line)
        if match and self.current_game:
            self.current_game.warnings.append(match.group(1))
            return
        
        # Errors
        match = self.ERROR.search(line)
        if match and self.current_game:
            self.current_game.errors.append(match.group(1))
            return
        
        # Phase transitions
        match = self.PHASE_TRANSITION.search(line)
        if match:
            transition = PhaseTransition(
                game_num=self.current_game.game_num if self.current_game else 0,
                from_phase=int(match.group(1)),
                to_phase=int(match.group(2)),
                old_rounds=int(match.group(3)),
                new_rounds=int(match.group(4)),
                old_actions=match.group(5),
                new_actions=match.group(6)
            )
            self.phase_transitions.append(transition)
            return
        
        # Model updates
        match = self.MODEL_UPDATE_START.search(line)
        if match:
            if self.current_update:
                self.model_updates.append(self.current_update)
            
            self.current_update = ModelUpdate(
                game_num=int(match.group(1)),
                agents_updated=[],
                buffer_sizes={},
                losses={}
            )
            return
        
        match = self.MODEL_UPDATE_AGENT.search(line)
        if match and self.current_update:
            self.current_update_agent = match.group(1)
            self.current_update.agents_updated.append(self.current_update_agent)
            return
        
        match = self.MODEL_BUFFER_SIZE.search(line)
        if match and self.current_update and self.current_update_agent:
            self.current_update.buffer_sizes[self.current_update_agent] = int(match.group(1))
            return
        
        match = self.MODEL_LOSSES.search(line)
        if match and self.current_update:
            agent = match.group(1)
            self.current_update.losses[agent] = {
                'policy_loss': float(match.group(2)),
                'value_loss': float(match.group(3)),
                'entropy': float(match.group(4))
            }
            return

print("✓ Parser class defined")

## Parse Log File

In [ ]:
# Parse the log file
if not LOG_FILE.exists():
    print(f"❌ Cannot proceed: {LOG_FILE} not found")
else:
    parser = LogParser(LOG_FILE)
    parser.parse()
    
    # Extract data
    games = parser.games
    phase_transitions = parser.phase_transitions
    model_updates = parser.model_updates
    
    print(f"\n✓ Ready for analysis")

## 1. Training Run Summary

In [ ]:
if games:
    start_game = games[0].game_num
    end_game = games[-1].game_num
    total_games = len(games)
    
    print("="*80)
    print("TRAINING LOG ANALYSIS SUMMARY")
    print("="*80)
    
    print(f"\n📄 Log File: {LOG_FILE.name}")
    print(f"📊 Games: {start_game} → {end_game} ({total_games} games)")
    
    # Phase distribution
    phase_counts = defaultdict(int)
    for game in games:
        if game.phase:
            phase_counts[game.phase] += 1
    
    print(f"\n📈 Curriculum Phases:")
    for phase in sorted(phase_counts.keys()):
        count = phase_counts[phase]
        pct = 100 * count / total_games
        max_rds = {1: 15, 2: 25, 3: 50}[phase]
        max_acts = {1: 10, 2: 20, 3: 50}[phase]
        print(f"   Phase {phase}: {count:4d} games ({pct:5.1f}%) - {max_rds} rds, {max_acts}× actions")
    
    # Phase transitions
    if phase_transitions:
        print(f"\n🔄 Phase Transitions: {len(phase_transitions)}")
        for trans in phase_transitions:
            print(f"   Game {trans.game_num}: Phase {trans.from_phase}→{trans.to_phase} "
                  f"({trans.old_rounds}→{trans.new_rounds} rds, {trans.old_actions}→{trans.new_actions})")
    
    # Game statistics
    completed_games = [g for g in games if g.steps is not None]
    if completed_games:
        avg_steps = np.mean([g.steps for g in completed_games])
        avg_rounds = np.mean([g.rounds for g in completed_games if g.rounds])
        avg_turns = np.mean([g.turns for g in completed_games if g.turns])
        
        print(f"\n📊 Game Statistics:")
        print(f"   Completed games: {len(completed_games)}/{total_games}")
        print(f"   Average steps/game: {avg_steps:.1f}")
        print(f"   Average rounds/game: {avg_rounds:.1f}")
        print(f"   Average turns/game: {avg_turns:.1f}")
    
    # Warnings and errors
    total_warnings = sum(len(g.warnings) for g in games)
    total_errors = sum(len(g.errors) for g in games)
    
    if total_warnings > 0 or total_errors > 0:
        print(f"\n⚠️  Issues Detected:")
        print(f"   Warnings: {total_warnings}")
        print(f"   Errors: {total_errors}")
        
        # Most common warnings
        all_warnings = defaultdict(int)
        for game in games:
            for warning in game.warnings:
                if "Turn exceeded" in warning:
                    all_warnings["Turn exceeded max actions"] += 1
                elif "reached max_rounds" in warning:
                    all_warnings["Reached max_rounds"] += 1
                elif "only one player remaining" in warning:
                    all_warnings["Only one player remaining"] += 1
                else:
                    all_warnings[warning[:50]] += 1
        
        if all_warnings:
            print(f"\n   Most common warnings:")
            for warning, count in sorted(all_warnings.items(), key=lambda x: -x[1])[:5]:
                print(f"      • {warning}: {count} occurrences")
    
    print("\n" + "="*80)
else:
    print("❌ No games found in log file")

## 2. Create DataFrame for Analysis

In [ ]:
if games:
    # Build DataFrame
    data = []
    for game in games:
        data.append({
            'game_num': game.game_num,
            'num_players': game.num_players,
            'phase': game.phase,
            'max_rounds': game.max_rounds,
            'max_actions_multiplier': game.max_actions_multiplier,
            'winner': game.winner,
            'rounds': game.rounds,
            'turns': game.turns,
            'steps': game.steps,
            'warning_count': len(game.warnings),
            'error_count': len(game.errors),
            'turn_exceeded_warnings': sum(1 for w in game.warnings if "Turn exceeded" in w),
            'max_rounds_reached': sum(1 for w in game.warnings if "reached max_rounds" in w)
        })
    
    df = pd.DataFrame(data)
    
    print(f"✓ Created DataFrame with {len(df)} games")
    print(f"\nFirst 10 games:")
    display(df.head(10))
    
    print(f"\nDataFrame info:")
    print(df.info())
    
    print(f"\nBasic statistics:")
    display(df.describe())
else:
    print("❌ No data to create DataFrame")

## 3. Game Length Progression

In [ ]:
if games and df is not None:
    fig, axes = plt.subplots(3, 1, figsize=(16, 14))
    
    # Phase colors
    phase_colors = {1: '#ff9999', 2: '#99ccff', 3: '#99ff99'}
    
    # Filter completed games
    df_complete = df[df['steps'].notna()].copy()
    
    # Plot 1: Steps per game
    ax1 = axes[0]
    if len(df_complete) > 0:
        colors = [phase_colors[p] for p in df_complete['phase']]
        ax1.scatter(df_complete['game_num'], df_complete['steps'], 
                   c=colors, alpha=0.6, s=30, edgecolors='none')
        
        # Rolling average
        if len(df_complete) > 50:
            rolling_steps = df_complete['steps'].rolling(window=50, min_periods=1).mean()
            ax1.plot(df_complete['game_num'], rolling_steps, 'b-', linewidth=2, 
                    label='50-game rolling avg', alpha=0.8)
        
        # Phase transitions
        for trans in phase_transitions:
            ax1.axvline(trans.game_num, color='red', linestyle='--', 
                       alpha=0.5, linewidth=2, label='Phase transition' if trans == phase_transitions[0] else '')
        
        ax1.set_xlabel('Game Number', fontsize=12)
        ax1.set_ylabel('Steps per Game', fontsize=12)
        ax1.set_title('Game Length (Steps) Over Training', fontsize=14, fontweight='bold')
        ax1.legend(loc='upper right')
        ax1.grid(True, alpha=0.3)
    
    # Plot 2: Rounds per game
    ax2 = axes[1]
    df_rounds = df[df['rounds'].notna()].copy()
    if len(df_rounds) > 0:
        colors = [phase_colors[p] for p in df_rounds['phase']]
        ax2.scatter(df_rounds['game_num'], df_rounds['rounds'], 
                   c=colors, alpha=0.6, s=30, edgecolors='none')
        
        # Rolling average
        if len(df_rounds) > 50:
            rolling_rounds = df_rounds['rounds'].rolling(window=50, min_periods=1).mean()
            ax2.plot(df_rounds['game_num'], rolling_rounds, 'b-', linewidth=2, 
                    label='50-game rolling avg', alpha=0.8)
        
        # Max rounds lines
        for phase in [1, 2, 3]:
            max_rds = {1: 15, 2: 25, 3: 50}[phase]
            ax2.axhline(max_rds, color=phase_colors[phase], linestyle=':', 
                       alpha=0.5, linewidth=2, label=f'Phase {phase} max ({max_rds})')
        
        # Phase transitions
        for trans in phase_transitions:
            ax2.axvline(trans.game_num, color='red', linestyle='--', alpha=0.5, linewidth=2)
        
        ax2.set_xlabel('Game Number', fontsize=12)
        ax2.set_ylabel('Rounds per Game', fontsize=12)
        ax2.set_title('Game Length (Rounds) Over Training', fontsize=14, fontweight='bold')
        ax2.legend(loc='upper right', fontsize=8)
        ax2.grid(True, alpha=0.3)
    
    # Plot 3: Turns per game
    ax3 = axes[2]
    df_turns = df[df['turns'].notna()].copy()
    if len(df_turns) > 0:
        colors = [phase_colors[p] for p in df_turns['phase']]
        ax3.scatter(df_turns['game_num'], df_turns['turns'], 
                   c=colors, alpha=0.6, s=30, edgecolors='none')
        
        # Rolling average
        if len(df_turns) > 50:
            rolling_turns = df_turns['turns'].rolling(window=50, min_periods=1).mean()
            ax3.plot(df_turns['game_num'], rolling_turns, 'b-', linewidth=2, 
                    label='50-game rolling avg', alpha=0.8)
        
        # Phase transitions
        for trans in phase_transitions:
            ax3.axvline(trans.game_num, color='red', linestyle='--', alpha=0.5, linewidth=2)
        
        ax3.set_xlabel('Game Number', fontsize=12)
        ax3.set_ylabel('Turns per Game', fontsize=12)
        ax3.set_title('Game Length (Turns) Over Training', fontsize=14, fontweight='bold')
        ax3.legend(loc='upper right')
        ax3.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
else:
    print("❌ No data to plot")

## 4. Warnings and Issues Timeline

In [ ]:
if games and df is not None:
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 10))
    
    # Plot 1: All warnings
    ax1.scatter(df['game_num'], df['warning_count'], alpha=0.5, s=40, color='orange')
    
    if len(df) > 50:
        rolling_warnings = df['warning_count'].rolling(window=50, min_periods=1).mean()
        ax1.plot(df['game_num'], rolling_warnings, 'r-', linewidth=2.5, 
                label='50-game rolling avg', alpha=0.8)
    
    for trans in phase_transitions:
        ax1.axvline(trans.game_num, color='red', linestyle='--', 
                   alpha=0.5, linewidth=2, label='Phase transition' if trans == phase_transitions[0] else '')
    
    ax1.set_xlabel('Game Number', fontsize=12)
    ax1.set_ylabel('Warnings per Game', fontsize=12)
    ax1.set_title('Total Warnings Over Training', fontsize=14, fontweight='bold')
    ax1.legend(loc='upper right')
    ax1.grid(True, alpha=0.3)
    
    # Plot 2: Specific warning types
    ax2.scatter(df['game_num'], df['turn_exceeded_warnings'], 
               alpha=0.6, s=40, label='Turn exceeded max actions', color='orange')
    ax2.scatter(df['game_num'], df['max_rounds_reached'], 
               alpha=0.6, s=40, label='Max rounds reached', color='purple')
    
    for trans in phase_transitions:
        ax2.axvline(trans.game_num, color='red', linestyle='--', 
                   alpha=0.5, linewidth=2)
    
    ax2.set_xlabel('Game Number', fontsize=12)
    ax2.set_ylabel('Count per Game', fontsize=12)
    ax2.set_title('Specific Warning Types Over Training', fontsize=14, fontweight='bold')
    ax2.legend(loc='upper right')
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Summary statistics
    print("\n📊 Warning Statistics:")
    print(f"   Games with warnings: {len(df[df['warning_count'] > 0])}/{len(df)}")
    print(f"   Average warnings per game: {df['warning_count'].mean():.2f}")
    print(f"   Max warnings in a game: {df['warning_count'].max()}")
    print(f"   Games with 'Turn exceeded': {len(df[df['turn_exceeded_warnings'] > 0])}")
    print(f"   Games reaching max_rounds: {len(df[df['max_rounds_reached'] > 0])}")
else:
    print("❌ No data to plot")

## 5. Phase-by-Phase Comparison

In [ ]:
if games and df is not None:
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    # Group by phase
    phase_groups = df.groupby('phase')
    
    # Plot 1: Average steps by phase
    ax1 = axes[0, 0]
    phase_stats = df.groupby('phase')['steps'].agg(['mean', 'std', 'count'])
    if len(phase_stats) > 0:
        x = phase_stats.index
        y = phase_stats['mean']
        err = phase_stats['std']
        
        bars = ax1.bar(x, y, yerr=err, capsize=5, alpha=0.7, 
                      color=['#ff9999', '#99ccff', '#99ff99'][:len(x)])
        
        # Add value labels
        for i, (idx, bar) in enumerate(zip(x, bars)):
            height = bar.get_height()
            ax1.text(bar.get_x() + bar.get_width()/2., height,
                    f'{y.iloc[i]:.0f}\n(n={phase_stats["count"].iloc[i]})',
                    ha='center', va='bottom', fontsize=10)
        
        ax1.set_xlabel('Curriculum Phase', fontsize=12)
        ax1.set_ylabel('Average Steps per Game', fontsize=12)
        ax1.set_title('Average Game Length by Phase', fontsize=13, fontweight='bold')
        ax1.set_xticks(x)
        ax1.grid(True, alpha=0.3, axis='y')
    
    # Plot 2: Average rounds by phase
    ax2 = axes[0, 1]
    phase_rounds = df.groupby('phase')['rounds'].agg(['mean', 'std', 'count'])
    if len(phase_rounds) > 0:
        x = phase_rounds.index
        y = phase_rounds['mean']
        
        bars = ax2.bar(x, y, alpha=0.7, color=['#ff9999', '#99ccff', '#99ff99'][:len(x)])
        
        # Add max_rounds lines
        for phase_num in x:
            max_rds = {1: 15, 2: 25, 3: 50}[phase_num]
            ax2.axhline(max_rds, color='red', linestyle='--', alpha=0.5, linewidth=1.5)
            ax2.text(phase_num, max_rds, f' max={max_rds}', 
                    va='bottom', fontsize=9, color='red')
        
        # Add value labels
        for i, (idx, bar) in enumerate(zip(x, bars)):
            height = bar.get_height()
            ax2.text(bar.get_x() + bar.get_width()/2., height,
                    f'{y.iloc[i]:.1f}',
                    ha='center', va='bottom', fontsize=10)
        
        ax2.set_xlabel('Curriculum Phase', fontsize=12)
        ax2.set_ylabel('Average Rounds per Game', fontsize=12)
        ax2.set_title('Average Rounds by Phase (vs max_rounds)', fontsize=13, fontweight='bold')
        ax2.set_xticks(x)
        ax2.grid(True, alpha=0.3, axis='y')
    
    # Plot 3: Warning rate by phase
    ax3 = axes[1, 0]
    phase_warnings = df.groupby('phase')['warning_count'].agg(['mean', 'sum', 'count'])
    if len(phase_warnings) > 0:
        x = phase_warnings.index
        y = phase_warnings['mean']
        
        bars = ax3.bar(x, y, alpha=0.7, color=['#ff9999', '#99ccff', '#99ff99'][:len(x)])
        
        # Add value labels
        for i, (idx, bar) in enumerate(zip(x, bars)):
            height = bar.get_height()
            total = phase_warnings['sum'].iloc[i]
            ax3.text(bar.get_x() + bar.get_width()/2., height,
                    f'{y.iloc[i]:.2f}\n({int(total)} total)',
                    ha='center', va='bottom', fontsize=10)
        
        ax3.set_xlabel('Curriculum Phase', fontsize=12)
        ax3.set_ylabel('Average Warnings per Game', fontsize=12)
        ax3.set_title('Warning Rate by Phase', fontsize=13, fontweight='bold')
        ax3.set_xticks(x)
        ax3.grid(True, alpha=0.3, axis='y')
    
    # Plot 4: Distribution of game lengths by phase
    ax4 = axes[1, 1]
    for phase in sorted(df['phase'].unique()):
        phase_data = df[df['phase'] == phase]['steps'].dropna()
        if len(phase_data) > 0:
            ax4.hist(phase_data, bins=30, alpha=0.5, 
                    label=f'Phase {phase} (n={len(phase_data)})',
                    color=['#ff9999', '#99ccff', '#99ff99'][phase-1])
    
    ax4.set_xlabel('Steps per Game', fontsize=12)
    ax4.set_ylabel('Frequency', fontsize=12)
    ax4.set_title('Distribution of Game Lengths by Phase', fontsize=13, fontweight='bold')
    ax4.legend(loc='upper right')
    ax4.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()
else:
    print("❌ No data to plot")

## 6. Model Update Tracking

In [ ]:
if model_updates:
    print(f"\n📊 Model Updates: {len(model_updates)} total")
    print(f"   Update frequency: every {model_updates[1].game_num - model_updates[0].game_num if len(model_updates) > 1 else 'N/A'} games")
    
    # Extract loss data
    update_data = []
    for update in model_updates:
        for agent, losses in update.losses.items():
            update_data.append({
                'game_num': update.game_num,
                'agent': agent,
                'policy_loss': losses['policy_loss'],
                'value_loss': losses['value_loss'],
                'entropy': losses['entropy']
            })
    
    if update_data:
        df_updates = pd.DataFrame(update_data)
        
        # Plot losses over time
        fig, axes = plt.subplots(3, 1, figsize=(16, 12))
        
        # Get unique agents
        agents = df_updates['agent'].unique()
        
        # Plot 1: Policy loss
        ax1 = axes[0]
        for agent in agents:
            agent_data = df_updates[df_updates['agent'] == agent]
            ax1.plot(agent_data['game_num'], agent_data['policy_loss'], 
                    marker='o', label=agent, alpha=0.7, linewidth=1.5)
        
        ax1.set_xlabel('Game Number', fontsize=12)
        ax1.set_ylabel('Policy Loss', fontsize=12)
        ax1.set_title('Policy Loss Over Training', fontsize=14, fontweight='bold')
        ax1.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
        ax1.grid(True, alpha=0.3)
        
        # Plot 2: Value loss
        ax2 = axes[1]
        for agent in agents:
            agent_data = df_updates[df_updates['agent'] == agent]
            ax2.plot(agent_data['game_num'], agent_data['value_loss'], 
                    marker='o', label=agent, alpha=0.7, linewidth=1.5)
        
        ax2.set_xlabel('Game Number', fontsize=12)
        ax2.set_ylabel('Value Loss', fontsize=12)
        ax2.set_title('Value Loss Over Training', fontsize=14, fontweight='bold')
        ax2.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
        ax2.grid(True, alpha=0.3)
        
        # Plot 3: Entropy
        ax3 = axes[2]
        for agent in agents:
            agent_data = df_updates[df_updates['agent'] == agent]
            ax3.plot(agent_data['game_num'], agent_data['entropy'], 
                    marker='o', label=agent, alpha=0.7, linewidth=1.5)
        
        ax3.set_xlabel('Game Number', fontsize=12)
        ax3.set_ylabel('Entropy', fontsize=12)
        ax3.set_title('Policy Entropy Over Training', fontsize=14, fontweight='bold')
        ax3.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
        ax3.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        # Summary table
        print("\n📊 Latest Update Statistics:")
        latest_update = model_updates[-1]
        print(f"   Game: {latest_update.game_num}")
        print(f"   Agents updated: {len(latest_update.agents_updated)}")
        
        if latest_update.losses:
            loss_df = pd.DataFrame([
                {'Agent': agent, **losses} 
                for agent, losses in latest_update.losses.items()
            ])
            display(loss_df)
    else:
        print("   No loss data available")
else:
    print("\n⚠️  No model updates found in log")

## 7. Export Data

In [ ]:
if df is not None:
    # Create output directory
    output_dir = Path('analysis_output')
    output_dir.mkdir(exist_ok=True)
    
    # Export main DataFrame
    csv_path = output_dir / f"{LOG_FILE.stem}_analysis.csv"
    df.to_csv(csv_path, index=False)
    print(f"✓ Exported game data to: {csv_path}")
    
    # Export summary statistics by phase
    summary_path = output_dir / f"{LOG_FILE.stem}_phase_summary.csv"
    phase_summary = df.groupby('phase').agg({
        'game_num': 'count',
        'steps': ['mean', 'std', 'min', 'max'],
        'rounds': ['mean', 'std'],
        'turns': ['mean', 'std'],
        'warning_count': ['sum', 'mean'],
        'turn_exceeded_warnings': 'sum',
        'max_rounds_reached': 'sum'
    })
    phase_summary.to_csv(summary_path)
    print(f"✓ Exported phase summary to: {summary_path}")
    
    # Export model update data if available
    if model_updates and 'df_updates' in locals():
        updates_path = output_dir / f"{LOG_FILE.stem}_model_updates.csv"
        df_updates.to_csv(updates_path, index=False)
        print(f"✓ Exported model updates to: {updates_path}")
    
    print(f"\n📁 All outputs saved to: {output_dir.absolute()}")
else:
    print("❌ No data to export")

## 8. Key Insights & Recommendations

In [ ]:
if df is not None and len(df) > 0:
    print("="*80)
    print("KEY INSIGHTS & RECOMMENDATIONS")
    print("="*80)
    
    # 1. Learning Progress
    print("\n1️⃣  Learning Progress:")
    if len(df) > 100:
        early_games = df.head(50)
        recent_games = df.tail(50)
        
        early_steps = early_games['steps'].mean()
        recent_steps = recent_games['steps'].mean()
        
        if not np.isnan(early_steps) and not np.isnan(recent_steps):
            change_pct = ((recent_steps - early_steps) / early_steps) * 100
            
            if change_pct < -10:
                print(f"   ✅ Games are getting shorter: {early_steps:.0f} → {recent_steps:.0f} steps ({change_pct:.1f}%)")
                print(f"      This suggests agents are learning more efficient strategies")
            elif change_pct > 10:
                print(f"   ⚠️  Games are getting longer: {early_steps:.0f} → {recent_steps:.0f} steps (+{change_pct:.1f}%)")
                print(f"      This may indicate agents struggling with new complexity")
            else:
                print(f"   → Game length stable: {early_steps:.0f} → {recent_steps:.0f} steps ({change_pct:+.1f}%)")
    
    # 2. Warning Trends
    print("\n2️⃣  Warning Trends:")
    if df['warning_count'].sum() > 0:
        if len(df) > 100:
            early_warnings = df.head(50)['warning_count'].mean()
            recent_warnings = df.tail(50)['warning_count'].mean()
            
            if recent_warnings < early_warnings * 0.5:
                print(f"   ✅ Warnings decreasing: {early_warnings:.2f} → {recent_warnings:.2f} per game")
                print(f"      Agents are learning to play within constraints")
            elif recent_warnings > early_warnings * 1.5:
                print(f"   ⚠️  Warnings increasing: {early_warnings:.2f} → {recent_warnings:.2f} per game")
                print(f"      Consider adjusting action limits or investigating stuck patterns")
        
        # Most problematic warning type
        turn_exceeded_total = df['turn_exceeded_warnings'].sum()
        max_rounds_total = df['max_rounds_reached'].sum()
        
        if turn_exceeded_total > max_rounds_total:
            print(f"   📌 Main issue: Turn exceeded warnings ({turn_exceeded_total} times)")
            print(f"      Agents may need more training to learn END_TURN action")
        elif max_rounds_total > 0:
            print(f"   📌 Main issue: Max rounds reached ({max_rounds_total} times)")
            print(f"      Games running long - consider if this is expected for current phase")
    else:
        print(f"   ✅ No warnings detected - clean training run!")
    
    # 3. Phase Transitions
    if phase_transitions:
        print("\n3️⃣  Phase Transition Effects:")
        for trans in phase_transitions:
            # Get games around transition
            before_games = df[(df['game_num'] >= trans.game_num - 50) & 
                             (df['game_num'] < trans.game_num)]
            after_games = df[(df['game_num'] >= trans.game_num) & 
                            (df['game_num'] < trans.game_num + 50)]
            
            if len(before_games) > 0 and len(after_games) > 0:
                before_warnings = before_games['warning_count'].mean()
                after_warnings = after_games['warning_count'].mean()
                
                print(f"   Phase {trans.from_phase}→{trans.to_phase} at game {trans.game_num}:")
                if after_warnings > before_warnings * 1.2:
                    print(f"      ⚠️  Warning spike: {before_warnings:.2f} → {after_warnings:.2f}")
                    print(f"         Expected during adaptation period")
                else:
                    print(f"      ✅ Smooth transition: {before_warnings:.2f} → {after_warnings:.2f}")
    
    # 4. Current Status
    print("\n4️⃣  Current Training Status:")
    last_game = df.iloc[-1]
    print(f"   Last game: {int(last_game['game_num'])}")
    print(f"   Current phase: {int(last_game['phase'])}")
    
    phase = int(last_game['phase'])
    if phase == 1:
        remaining = 1000 - last_game['game_num']
        print(f"   Games until Phase 2: {int(remaining)}")
    elif phase == 2:
        remaining = 3000 - last_game['game_num']
        print(f"   Games until Phase 3: {int(remaining)}")
    else:
        print(f"   In final phase - full complexity")
    
    # 5. Recommendations
    print("\n5️⃣  Recommendations:")
    
    recent_warnings = df.tail(50)['warning_count'].mean()
    if recent_warnings > 2:
        print(f"   • High warning rate ({recent_warnings:.1f}/game) - consider:")
        print(f"     - Reviewing reward shaping for END_TURN action")
        print(f"     - Checking if action masking is working correctly")
        print(f"     - Verifying agents aren't stuck in loops")
    
    if len(df) < 1000:
        print(f"   • Still in early training - continue building baseline")
    elif len(df) < 3000:
        print(f"   • Mid-training - watch for Phase 2 transition effects")
    
    completion_rate = len(df[df['steps'].notna()]) / len(df) * 100
    if completion_rate < 90:
        print(f"   • Game completion rate: {completion_rate:.1f}% - some games may be interrupted")
    
    print("\n" + "="*80)
else:
    print("❌ Insufficient data for insights")

---

## Next Steps

**For deeper analysis:**
1. Open `analyze_curriculum_training.ipynb` to analyze checkpoint histories (win rates, rewards)
2. Cross-reference findings between log analysis and checkpoint analysis
3. Use exported CSVs for custom analysis in pandas

**For continued monitoring:**
1. Re-run this notebook periodically during training
2. Compare multiple log files to track progress across sessions
3. Watch for trends in warning rates and game lengths